# Hospital Readmission Prediction — Machine Learning Pipeline
### Dataset: HLT-005 Synthetic Hospital Admission Dataset
**Course / Project:** Undergraduate Machine Learning Project  
**Target:** `readmit_flag_30d` (0 = Not Readmitted within 30 days, 1 = Readmitted within 30 days)  

---
### Project Objectives:
1. **Understand & Explore** the synthetic hospital admission data (EDA, missing values, distributions).
2. **Preprocess Data** safely with zero data leakage (patient-level grouping, median/mode imputation, scaling, one-hot encoding).
3. **Feature Selection & Engineering** selecting 7 clinically meaningful, explainable features.
4. **Train & Evaluate 3 Core Algorithms**:
   - Logistic Regression (interpretable linear baseline)
   - Decision Tree Classifier (rule-based white-box model)
   - Random Forest Classifier (ensemble of decision trees)
5. **Handle Class Imbalance** via cost-sensitive class weighting (`class_weight='balanced'`).
6. **Compare Models** using Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC.


In [ ]:
import os
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Libraries loaded successfully!')

## 1. Data Understanding & Exploratory Data Analysis (EDA)

In [ ]:
# Load HLT-005 synthetic dataset
data_path = Path('../data/admissions.csv')
if not data_path.exists():
    data_path = Path('data/admissions.csv')

df_raw = pd.read_csv(data_path)
print(f'Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Unique Patients: {df_raw["patient_id"].nunique():,}')
df_raw.head()

In [ ]:
# Check target class distribution
target_counts = df_raw['readmit_flag_30d'].value_counts()
print('Target Counts:')
print(target_counts)
print('\nTarget Proportions:')
print(df_raw['readmit_flag_30d'].value_counts(normalize=True))

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
bars = ax.bar(['Not Readmitted (0)', 'Readmitted (1)'], target_counts.values, color=['#176B87', '#E74C3C'], width=0.5)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{h:,} ({h/len(df_raw)*100:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 4), textcoords='offset points', ha='center', fontweight='bold')
ax.set_title('Target Class Distribution (30-Day Readmission)', fontweight='bold')
ax.set_ylabel('Encounters')
plt.tight_layout()
plt.show()

## 2. Feature Selection & Clinical Mapping
We select **7 meaningful, explainable clinical features** that are available at prediction time:
1. `age`: Patient age in years
2. `prior_admits_12mo`: Inpatient admissions in prior 12 months
3. `prior_ed_visits_6mo`: Emergency department visits in prior 6-12 months
4. `los_days`: Length of hospital stay in days
5. `cci_score`: Charlson Comorbidity Index score
6. `diagnosis_category`: Primary diagnosis category (mapped from MS-DRG)
7. `icu_flag_str`: ICU admission indicator ('No' / 'Yes')


In [ ]:
def map_diagnosis(drg_label):
    drg = str(drg_label).lower()
    if 'septicemia' in drg or 'sepsis' in drg:
        return 'Infection/Sepsis'
    elif 'heart' in drg or 'cardio' in drg or 'chest pain' in drg:
        return 'Cardiovascular'
    elif 'pneumonia' in drg or 'respiratory' in drg:
        return 'Respiratory'
    elif 'hemorrhage' in drg or 'stroke' in drg or 'seizure' in drg:
        return 'Neurological'
    elif 'joint' in drg or 'spinal' in drg or 'fracture' in drg or 'deran' in drg:
        return 'Orthopedic/Musculoskeletal'
    elif 'renal' in drg:
        return 'Renal'
    elif 'esophagitis' in drg or 'gi' in drg:
        return 'Digestive/GI'
    elif 'delivery' in drg or 'newborn' in drg:
        return 'Obstetric/Newborn'
    elif 'psychoses' in drg or 'alcohol' in drg or 'drug' in drg:
        return 'Psychiatric/Substance'
    else:
        return 'Other/Symptoms'

df = df_raw.copy()
df['diagnosis_category'] = df['ms_drg_label'].apply(map_diagnosis)
df['icu_flag_str'] = df['icu_flag'].map({0: 'No', 1: 'Yes'})

features = ['age', 'prior_admits_12mo', 'prior_ed_visits_6mo', 'los_days', 'cci_score', 'diagnosis_category', 'icu_flag_str']
num_cols = ['age', 'prior_admits_12mo', 'prior_ed_visits_6mo', 'los_days', 'cci_score']
cat_cols = ['diagnosis_category', 'icu_flag_str']
target_col = 'readmit_flag_30d'

print('Diagnosis Categories:')
print(df['diagnosis_category'].value_counts())

## 3. Data Preprocessing & Patient-Level Split
To prevent **patient data leakage** across multiple hospital admissions, we use `StratifiedGroupKFold` grouped by `patient_id`. All records of any given patient stay strictly within either the training or testing partition.


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(sgkf.split(df, df[target_col], groups=df['patient_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

overlap = len(set(train_df['patient_id']).intersection(set(test_df['patient_id'])))
print(f'Training encounters: {len(train_df):,} | Readmission rate: {train_df[target_col].mean()*100:.2f}%')
print(f'Testing encounters:  {len(test_df):,} | Readmission rate: {test_df[target_col].mean()*100:.2f}%')
print(f'Patient overlap between Train & Test: {overlap} (ZERO leakage confirmed ✅)')

X_train = train_df[features]
y_train = train_df[target_col]
X_test = test_df[features]
y_test = test_df[target_col]

In [ ]:
# Scikit-Learn Preprocessing Pipeline
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Fit on training data ONLY
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f'Transformed feature shape: Train={X_train_proc.shape}, Test={X_test_proc.shape}')

## 4. Model Training & Evaluation
We train and compare three distinct algorithms:
1. **Logistic Regression** (Linear baseline with balanced class weights)
2. **Decision Tree** (Interpretable tree with max_depth=4)
3. **Random Forest** (150-tree ensemble)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=4, min_samples_leaf=20, random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=150, max_depth=5, min_samples_leaf=15, random_state=RANDOM_SEED)
}

results = []
cms = {}
for name, clf in models.items():
    clf.fit(X_train_proc, y_train)
    y_pred = clf.predict(X_test_proc)
    y_proba = clf.predict_proba(X_test_proc)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    cms[name] = cm
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    })

eval_df = pd.DataFrame(results)
eval_df

## 5. Model Evaluation Visualizations

In [ ]:
# Model Comparison Bar Chart
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
melted = eval_df.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'], var_name='Metric', value_name='Score')
sns.barplot(data=melted, x='Metric', y='Score', hue='Model', palette='Blues_r', ax=ax, edgecolor='#123650')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
ax.set_ylim(0, 1.0)
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f'{h:.2f}', xy=(p.get_x() + p.get_width()/2, h), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
colors = {'Logistic Regression': '#176B87', 'Decision Tree': '#F39C12', 'Random Forest': '#E74C3C'}

# ROC Curves
for name, clf in models.items():
    y_proba = clf.predict_proba(X_test_proc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax1.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=colors[name], lw=2)

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Guess')
ax1.set_title('Receiver Operating Characteristic (ROC)', fontweight='bold')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate (Recall)')
ax1.legend(loc='lower right')

# PR Curves
for name, clf in models.items():
    y_proba = clf.predict_proba(X_test_proc)[:, 1]
    p, r, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax2.plot(r, p, label=f'{name} (PR-AUC={ap:.3f})', color=colors[name], lw=2)

ax2.axhline(y_test.mean(), color='k', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean()*100:.1f}%)')
ax2.set_title('Precision-Recall Curves', fontweight='bold')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Random Forest Feature Importance
rf = models['Random Forest']
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols).tolist()
feat_names = num_cols + cat_feature_names

display_names = [f.replace('diagnosis_category_', 'Dx: ').replace('icu_flag_str_Yes', 'ICU: Yes') for f in feat_names]
feat_imp = pd.Series(rf.feature_importances_, index=display_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
feat_imp.plot(kind='barh', color='#176B87', edgecolor='#123650', ax=ax)
ax.set_title('Random Forest — Feature Importance', fontweight='bold', fontsize=13)
ax.set_xlabel('Relative Importance')
plt.tight_layout()
plt.show()